In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Hello! Explain what is API clearly with an images also"
)

print(response.text)


An **API** stands for **Application Programming Interface**. 

Simply put, an API is a **messenger** that takes a request from you, tells a system what you want to do, and then brings the response back to you.

---

### 💡 The Best Way to Understand API: The Restaurant Analogy

Imagine you are sitting at a table in a restaurant.

1. **You** are the **App** (User).
2. The **Kitchen** is the **Server / Database** (where the data/food is stored).
3. The **Waiter** is the **API**.

```text
  +-------------+              +-------------+              +-------------+
  |             |  1. Order    |             |  2. Order    |             |
  |  YOU (App)  | -----------> |   WAITER    | -----------> |   KITCHEN   |
  |             |              |    (API)    |              |  (Server)   |
  |             |  4. Food     |             |  3. Food     |             |
  |             | <----------- |             | <----------- |             |
  +-------------+              +-------------+        

In [ ]:
def calculator(a:int,b:int)->int:
  """Add two numbers and return the result."""
  return a+b

In [ ]:
# Provide tools to Gemini.
from google import genai
from google.genai import types
from google.colab import userdata
# Retrieve the API key from Colab Secrets.
api_key=userdata.get("GEMINI_API_KEY")
# Create a Gemini client.
client=genai.Client(api_key=api_key)
# Tool 1 : Calculator
def calculator(a:int,b:int) -> int:
  """Add two numbers and return the result"""
  return a+b
  # Tool2 : Greeting
def greeting() ->str:
    """Greet the user with a welcome message."""
    return "Hello Sai! Welcome to our AI Agent"
# Tool 3: Weather
def weather() ->str:
    """Return sample weather information"""
    return "The weather is sunny today."
# Send the request to Gemini.
response=client.models.generate_content(
    model="gemini-2.5-flash",
    contents="""
    complete all the tasks using the availble tools:
    1.what is 10+20?use the calculator tool?
    2.Greet me using the greeting tool,
    3.Tell me the weather using the weather tool
    4.show me current weather report
    Give all three results in one response wth seperate headings
    """,
    config=types.GenerateContentConfig(
        tools=[calculator,
               greeting,
               weather]
    )
)

print(response.text)


### 1. Calculation Result
**10 + 20 = 30**

### 2. Greeting
Hello Sai! Welcome to our AI Agent

### 3. Current Weather Report
The weather is sunny today.


In [ ]:
# Project day: Python.
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

class GeminiAgent:

    def __init__(self, client, tools):
        self.client = client
        self.tools = tools

    def run(self, user_message):

        response = self.client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=user_message,
            config=types.GenerateContentConfig(
                tools=self.tools,
                automatic_function_calling=types.AutomaticFunctionCallingConfig( # The SDK automatically executes a tool when Gemini requests a function call.
                    disable=False # automatic function call enabled
                )
            )
        )

        return response.text


agent = GeminiAgent(
    client=client,
    tools=[calculator, greeting, weather]
)

print(agent.run("What is 100 multiplied by 50? Use calculator."))

100 multiplied by 50 is 5000.


In [ ]:
# Tool schema.
def calculator(a: int, b: int) -> int:
    """Add two numbers and return the result."""
    return a + b
# Gemini needs the tool's purpose, name, required arguments, argument data types, and expected result.
# Upgrade the calculator tool.
def calculator(a: float, b: float, operation: str) -> float:
    """
    Perform a mathematical operation on two numbers.
    #The Gemini SDK uses the function definition, type annotations, and docstring to help define the tool schema.
    Args:
        a: First number.
        b: Second number.
        operation: Operation to perform. Options are add, subtract, multiply, divide.

    Returns:
        The calculated result.
    """

    if operation == "add":
        return a + b

    elif operation == "subtract":
        return a - b

    elif operation == "multiply":
        return a * b

    elif operation == "divide":
        if b == 0:
            return "Cannot divide by zero"

        return a / b

    else:
        return "Invalid operation"
# Connect this tool to Gemini.
agent=GeminiAgent(
    client=client,
    tools=[calculator,greeting,weather]
)
print(agent.run("Multiply 25 and 40 using calculator"))
print(agent.run("explain complte about computer in ur own words ."))

The result of multiplying 25 and 40 is 1000.
At its core, a **computer** is an electronic device that takes in raw information, processes it according to a set of instructions, and gives you a useful result. 

Think of it like a very smart, tireless digital assistant that can do billions of calculations in a fraction of a second, but it can only do what humans tell it to do.

To understand a computer completely, let's break it down into four main parts: **Hardware**, **Software**, **How it works (The Cycle)**, and **Why we use it**.

---

### 1. The Hardware (The Body)
Hardware refers to all the physical, tangible parts of the computer that you can actually touch. It is divided into four main categories:

*   **Input Devices:** These are how you talk to the computer and feed it data.
    *   *Examples:* Keyboard, mouse, microphone, webcam, touchscreen.
*   **Processing Unit (CPU - Central Processing Unit):** Often called the "brain" of the computer. It fetches instructions from memory,

In [ ]:
# Automatic tool calling: The Gemini SDK executes requested tools and returns their results automatically.
# Manual tool-calling flow: The user sends a request to Gemini; Gemini returns a function call; Python identifies and executes the function; the result is sent back to Gemini; Gemini generates the final answer.
from google import genai
from google.genai import types
# Map tool names to Python functions.
tool_functions ={
    "calculator":calculator,
     "greeting": greeting,
     "weather": weather
}
def manual_agent(user_message):
    # Ask Gemini to decide whether a tool is needed.
    response=client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=user_message,
        config=types.GenerateContentConfig(
            tools=[calculator, greeting, weather],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(
                disable=True # Disable automatic function execution so Python can call the tool manually.
            )
        )
    )
    # Check the Gemini response for function calls.
    function_calls=response.function_calls # Check whether Gemini decided to call a tool.
    if not function_calls:  # If no function call is needed, return Gemini's response directly.
        return response.text

    # Execute tool calls manually
    function_response_parts = []

    for call in function_calls:
        print("Tool selected:", call.name)
        print("Arguments:", call.args)

        try:
            if call.name not in tool_functions:
                result = f"Error: {call.name} tool is not available."
                function_response_parts.append(
                    types.Part.from_function_response(
                        name=call.name,
                        response={"error": result}
                    )
                )
                print(result)
                continue

            function_to_call = tool_functions[call.name]
            result = function_to_call(**call.args)
            print("Tool result:", result)

            function_response_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )
        except Exception as e:
            print("Error occurred:", e)

    # Send tool result back to Gemini
    final_response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=[
            user_message,
            response.candidates[0].content,
            types.Content(
                role="user",
                parts=function_response_parts
            )
        ],
        config=types.GenerateContentConfig(
            tools=[calculator, greeting, weather]
        )
    )

    return final_response.text

print(manual_agent("what is 1000 multiplied by 50? use calculator"))
print(manual_agent("what is the weather condition tell me ...?"))
print(manual_agent("tell me greeting to anyone?"))
# Multiple tool calling.
print(manual_agent("first calculate 100 + 200 using the calculator"
"then greet to anyone using the greet tool"))
print(manual_agent("call the tool email"))

Tool selected: calculator
Arguments: {'a': 1000, 'b': 50, 'operation': 'multiply'}
Tool result: 50000
1000 multiplied by 50 is 50,000.
Tool selected: weather
Arguments: {}
Tool result: The weather is sunny today.
The weather is sunny today.
Tool selected: greeting
Arguments: {}
Tool result: Hello Sai! Welcome to our AI Agent
Hello Sai! Welcome to our AI Agent
Tool selected: calculator
Arguments: {'operation': 'add', 'a': 100, 'b': 200}
Tool result: 300
Tool selected: greeting
Arguments: {}
Tool result: Hello Sai! Welcome to our AI Agent
The result of 100 + 200 is 300. 

Hello Sai! Welcome to our AI Agent
I'm sorry, but I don't have an "email" tool available. Is there anything else I can help you with?


In [ ]:
# Tool argument validation and error handling.
def calculator(a: float, b: float, operation: str) -> float:
    # Validate numbers
    if not isinstance(a, (int, float)):
        raise ValueError("a must be a number")

    if not isinstance(b, (int, float)):
        raise ValueError("b must be a number")

    # Validate operation
    valid_operations = ["add", "subtract", "multiply", "divide"]
    if operation not in valid_operations:
        raise ValueError("Invalid operation")

    # Perform calculation
    if operation == "add":
        return a + b

    elif operation == "subtract":
        return a - b

    elif operation == "multiply":
        return a * b

    elif operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

print(calculator(10, 20, "multiply"))
try:
    print(calculator(10, 0, "divide"))
except ValueError as e:
    print("Error:", e)

try:
    print(calculator(10, 20, "power"))
except ValueError as e:
    print("Error:", e)

try:
    print(calculator("hello", 20, "add"))
except ValueError as e:
    print("Error:", e)

200
Error: Cannot divide by zero
Error: Invalid operation
Error: a must be a number


In [ ]:
# Tool argument validation.
# If Gemini sends incorrect inputs, Python must validate whether they are correct.
import time

# Test 1: Calculator
print("--- Test 1 ---")
print(manual_agent("Multiply 10 and 20 using calculator"))
time.sleep(10) # Increased delay to avoid 15 Requests Per Minute limit

# Test 2: Greeting
print("\n--- Test 2 ---")
print(manual_agent("Greet Sai using greeting tool"))
time.sleep(10)

# Test 3: Weather
print("\n--- Test 3 ---")
print(manual_agent("Tell me weather in Hyderabad"))
time.sleep(10)

# Test 4: Multiple tools
print("\n--- Test 4 ---")
print(manual_agent(
    "Calculate 100 + 200 using calculator "
    "and greet Sai using greeting tool"
))
time.sleep(10)

# Test 5: Unknown tool
print("\n--- Test 5 ---")
print(manual_agent("Send an email to Sai"))
time.sleep(10)

# Test 6: Normal question without tools
print("\n--- Test 6 ---")
print(manual_agent("What is an AI Agent?"))

--- Test 1 ---
Tool selected: calculator
Arguments: {'operation': 'multiply', 'a': 10, 'b': 20}
Tool result: 200
The result of multiplying 10 and 20 is 200.

--- Test 2 ---
Tool selected: greeting
Arguments: {}
Tool result: Hello Sai! Welcome to our AI Agent
Hello Sai! Welcome to our AI Agent

--- Test 3 ---
Tool selected: weather
Arguments: {}
Tool result: The weather is sunny today.
The weather in Hyderabad today is sunny!

--- Test 4 ---
Tool selected: calculator
Arguments: {'operation': 'add', 'b': 200, 'a': 100}
Tool result: 300
Tool selected: greeting
Arguments: {}
Tool result: Hello Sai! Welcome to our AI Agent
The result of 100 + 200 is 300, and here is your greeting: Hello Sai! Welcome to our AI Agent

--- Test 5 ---
I cannot send emails directly because I don't have access to your email account or contact list. However, if you'd like, I can help you draft the email! Just let me know what you'd like it to say.

--- Test 6 ---
An **AI Agent** (Artificial Intelligence Agent) is 